# E3 · Levers and drivers with explainable boosting, beside logistic regression

The planner's levers and the regional drivers of notebook 13, computed with explainable
boosting as well as logistic regression. Both are additive, so both split a region's gap
exactly; the difference is that explainable boosting's curves can bend. The table at the end
shows each lever's weight under both models and the gap: where they agree, the answer does not
depend on the shape of the model; where they part, it does, and the lever should be read with
care.

In [ ]:
# Settings for this notebook
QUICK = False        # True: a smoke run with 3 search trials per model; numbers will not match the study
FOLLOW_STUDY = True  # when a re-derived choice differs from the study's, carry the study's forward

import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd()
sys.path.insert(0, str(here.parent / "src" if here.name.startswith("notebooks") else here / "src"))
from anc8 import repro
P = repro.setup(quick=QUICK)
QUICK = P.quick

import numpy as np
import pandas as pd
from anc8 import library as lib, tuning

In [ ]:
repro.run_script("run_scenarios.py", "--models", "ebm", "logistic", *(["--quick"] if QUICK else []))
repro.run_script("run_regional_eda.py", "--models", "ebm", "logistic")

In [ ]:
s = json.load(open(P.outputs / "scenarios" / "scenarios.json"))
keys = [l["key"] for l in s["levers"]] + ["all seven"]
rows = []
for r in ("anc1", "anc4", "anc8"):
    e = s["by_model"]["ebm"]["rungs"][r]["gain"][-1]
    l = s["by_model"]["logistic"]["rungs"][r]["gain"][-1]
    for k, a, b in zip(keys, e, l):
        rows.append({"rung": r, "lever": k, "EBM": a * 100, "logistic": b * 100, "gap": (a - b) * 100,
                     "same direction": (a > 0) == (b > 0)})
t = pd.DataFrame(rows)
print(t.round(2).to_string(index=False))
print(f"\nThe models agree on direction for {t['same direction'].mean():.0%} of lever-rung pairs.")
ref = repro.reference("levers_national.json")
for m in ("ebm", "logistic"):
    got = {r: s["by_model"][m]["rungs"][r]["gain"][-1] for r in ref[m]}
    gap = max(abs(a - b) for r in ref[m] for a, b in zip(ref[m][r], got[r]))
    repro.check(f"the national lever effects ({m}) are the study's", gap < 1e-9, f"largest difference {gap:.1e}")
repro.report()